# Imports

In [1]:
import os
import time
import glob
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image
from sklearn.decomposition import PCA
from sklearn.random_projection import GaussianRandomProjection
from sklearn.metrics.pairwise import cosine_similarity
from IPython.display import display

In [14]:
import requests

# Choose a completely unique password-like topic name here
topic_name = "aysel_tfe_server_9988" 

# Configuration

In [2]:

# --- Configuration ---
CURRENT_DATASET = "Flickr8k"
DATA_DIR = os.path.join(os.getcwd(), 'TFE_Data')
IMAGE_DIR = os.path.join(DATA_DIR, "Flickr8k", "Images", "Flicker8k_Dataset")
IMAGE_PATHS = sorted(glob.glob(os.path.join(IMAGE_DIR, "*.jpg")))

DIMENSIONS_TO_TEST = [512, 256, 128, 64, 32, 16]

print(f"Environment ready. Target Dataset: {CURRENT_DATASET}")

Environment ready. Target Dataset: Flickr8k


# Data Loading

In [3]:
def load_raw_matrix(modality, model_name):
    """Loads the pre-computed RAW embedding matrix."""
    file_path = os.path.join(DATA_DIR, f"X_{modality}_{model_name}_raw_{CURRENT_DATASET}.npy")
    if not os.path.exists(file_path):
        raise FileNotFoundError(f"Matrix not found at {file_path}")
    return np.load(file_path)

In [4]:
def normalize_matrix(matrix):
    """L2 Normalization for Cosine Similarity."""
    norms = np.linalg.norm(matrix, axis=1, keepdims=True)
    return matrix / np.where(norms == 0, 1e-9, norms)

# Dimension Reducers

In [6]:
def apply_pca(raw_matrix, target_dim):
    return PCA(n_components=target_dim).fit_transform(raw_matrix)

In [7]:
def apply_grp(raw_matrix, target_dim):
    return GaussianRandomProjection(n_components=target_dim).fit_transform(raw_matrix)

# Green AI: Retrieval time analysis

In [8]:
def measure_retrieval_speed(matrix, num_trials=100):
    """Measures average milliseconds to query the entire database."""
    norm_mat = normalize_matrix(matrix)
    query = norm_mat[0].reshape(1, -1) # Random query
    
    start_time = time.perf_counter()
    for _ in range(num_trials):
        sims = cosine_similarity(query, norm_mat)
        _ = np.argsort(sims[0])[::-1]
    end_time = time.perf_counter()
    
    return ((end_time - start_time) / num_trials) * 1000

# Performance: Compare ground truth with reduced output

In [9]:
def evaluate_metrics(raw_sim, reduced_sim, gt_k=50, test_k=10):
    num_queries = raw_sim.shape[0]
    prec, rec, map_score = 0.0, 0.0, 0.0
    
    for i in range(num_queries):
        raw_ordered = np.argsort(raw_sim[i])[-(gt_k+1):-1][::-1]
        gt_set = set(raw_ordered)
        
        red_ordered = np.argsort(reduced_sim[i])[-(test_k+1):-1][::-1]
        retrieved_k = red_ordered[:test_k]
        
        hits = len(set(retrieved_k).intersection(gt_set))
        prec += hits / test_k
        rec += hits / gt_k
        
        ap, current_hits = 0.0, 0
        for rank, item_idx in enumerate(retrieved_k):
            if item_idx in gt_set:
                current_hits += 1
                ap += current_hits / (rank + 1.0)
        if len(gt_set) > 0:
            map_score += ap / min(len(gt_set), test_k)
            
    return {
        "Precision@10": (prec/num_queries)*100, 
        "Recall@10": (rec/num_queries)*100, 
        "mAP@10": (map_score/num_queries)*100
    }

# Run Pipeline

In [12]:
def run_master_pipeline(modality, model_name, reducer_name="PCA"):
    print(f"Running {reducer_name} Pipeline for {model_name.upper()} ({modality})")
    
    raw_matrix = load_raw_matrix(modality, model_name)
    max_dim = raw_matrix.shape[1]
    
    raw_norm = normalize_matrix(raw_matrix)
    raw_sim = cosine_similarity(raw_norm, raw_norm)
    raw_speed_ms = measure_retrieval_speed(raw_matrix)
    
    master_results = [{
        "Model": model_name, "Method": "RAW", "Dim": max_dim,
        "mAP@10": 100.00, "Prec@10": 100.00, "Rec@10": 100.00,
        "Time (ms)": raw_speed_ms, "Speedup": "1.0x"
    }]
    
    reduction_func = apply_pca if reducer_name == "PCA" else apply_grp
    
    for dim in DIMENSIONS_TO_TEST:
        if dim >= max_dim: continue
        
        reduced_matrix = reduction_func(raw_matrix, dim)
        red_norm = normalize_matrix(reduced_matrix)
        red_sim = cosine_similarity(red_norm, red_norm)
        
        speed_ms = measure_retrieval_speed(reduced_matrix)
        speedup = raw_speed_ms / speed_ms if speed_ms > 0 else 0
        
        metrics = evaluate_metrics(raw_sim, red_sim)
        
        master_results.append({
            "Model": model_name, "Method": reducer_name, "Dim": dim,
            "mAP@10": metrics["mAP@10"], "Prec@10": metrics["Precision@10"], 
            "Rec@10": metrics["Recall@10"],
            "Time (ms)": speed_ms, "Speedup": f"{speedup:.1f}x"
        })
        
    df = pd.DataFrame(master_results).round(2)
    display(df)
    
    # Generate LaTeX
    latex_str = df.to_latex(index=False, float_format="%.2f", 
                            caption=f"{reducer_name} Reduction Performance for {model_name.upper()}",
                            label=f"tab:dim_red_{model_name}_{reducer_name}")
    print(latex_str)
    
    return df, raw_matrix, raw_sim

# XAI: Visualiser

In [10]:
def xai_visualize_reduction(raw_matrix, raw_sim, query_index=42, target_dim=32, top_k=5, reducer_name="PCA"):
    print(f"Generating XAI Visualizer ({reducer_name}-{target_dim})")
    
    reduction_func = apply_pca if reducer_name == "PCA" else apply_grp
    reduced_matrix = reduction_func(raw_matrix, target_dim)
    
    query_reduced = reduced_matrix[query_index].reshape(1, -1)
    sims_reduced = cosine_similarity(query_reduced, reduced_matrix)[0]
    
    top_indices_raw = raw_sim[query_index].argsort()[-(top_k+1):][::-1][1:]
    top_indices_reduced = sims_reduced.argsort()[-(top_k+1):][::-1][1:]
    
    fig, axes = plt.subplots(2, top_k + 1, figsize=(15, 5))
    plt.subplots_adjust(wspace=0.1, hspace=0.3)
    
    query_img = Image.open(IMAGE_PATHS[query_index]).convert('RGB')
    axes[0, 0].imshow(query_img); axes[0, 0].set_title("QUERY (RAW)", fontweight="bold"); axes[0, 0].axis("off")
    axes[1, 0].imshow(query_img); axes[1, 0].set_title(f"QUERY ({reducer_name}-{target_dim})", fontweight="bold"); axes[1, 0].axis("off")
    
    for i, idx in enumerate(top_indices_raw):
        axes[0, i+1].imshow(Image.open(IMAGE_PATHS[idx]).convert('RGB'))
        axes[0, i+1].set_title(f"RAW Rank {i+1}\nSim: {raw_sim[query_index][idx]:.2f}", fontsize=9)
        axes[0, i+1].axis("off")
        
    for i, idx in enumerate(top_indices_reduced):
        border_color = "green" if idx in top_indices_raw else "red"
        match_text = "(MATCH)" if idx in top_indices_raw else "(NEW)"
        axes[1, i+1].imshow(Image.open(IMAGE_PATHS[idx]).convert('RGB'))
        axes[1, i+1].set_title(f"Rank {i+1} {match_text}\nSim: {sims_reduced[idx]:.2f}", fontsize=9, color=border_color)
        axes[1, i+1].axis("off")
        
    plt.show()

# Run experiment

In [16]:
VISION_MODELS = ["resnet50", "mobilenet_v3", "vit", "deit", "clip_vision"]
TEXT_MODELS = ["rnn", "bert", "roberta", "gpt2", "clip_text"]
REDUCERS = ["PCA", "GRP"]

all_results = []

# 1. Run all Vision Models
print("STARTING VISION EXPERIMENTS")
for model in VISION_MODELS:
    for reducer in REDUCERS:
        try:
            df, _, _ = run_master_pipeline(modality="vision", model_name=model, reducer_name=reducer)
            all_results.append(df)
            message = f"✅ TFE Server: Dimension Reduction Notebook has finished running on {model}!"
            requests.post(f"https://ntfy.sh/{topic_name}",  data=message.encode(encoding='utf-8'))
        except FileNotFoundError:
            print(f"Skipping {model} - Matrix not found.")

message = "✅ TFE Server: Dimension Reduction Notebook has finished running on vision models!"
requests.post(f"https://ntfy.sh/{topic_name}",  data=message.encode(encoding='utf-8'))


# 2. Run all Text Models
print("\nSTARTING TEXT EXPERIMENTS")
for model in TEXT_MODELS:
    for reducer in REDUCERS:
        try:
            df, _, _ = run_master_pipeline(modality="text", model_name=model, reducer_name=reducer)
            all_results.append(df)
            message = f"✅ TFE Server: Dimension Reduction Notebook has finished running on {model}!"
            requests.post(f"https://ntfy.sh/{topic_name}",  data=message.encode(encoding='utf-8'))
        except FileNotFoundError:
            print(f"Skipping {model} - Matrix not found.")
            
message = "✅ TFE Server: Dimension Reduction Notebook has finished running on text models!"
requests.post(f"https://ntfy.sh/{topic_name}",  data=message.encode(encoding='utf-8'))

# 3. Save a Master CSV with absolutely everything
if all_results:
    final_master_df = pd.concat(all_results, ignore_index=True)
    final_master_df.to_csv(os.path.join(DATA_DIR, "MASTER_THESIS_RESULTS.csv"), index=False)
    print("\nAll experiments complete! Saved to MASTER_THESIS_RESULTS.csv")

STARTING VISION EXPERIMENTS
Running PCA Pipeline for RESNET50 (vision)


,Model,Method,Dim,mAP@10,Prec@10,Rec@10,Time (ms),Speedup
0,resnet50,RAW,2048,100.00,100.00,100.00,8.14,1.0x
1,resnet50,PCA,512,98.40,98.63,19.73,5.18,1.6x
2,resnet50,PCA,256,97.54,97.96,19.59,3.22,2.5x
3,resnet50,PCA,128,95.95,96.66,19.33,0.77,10.6x
4,resnet50,PCA,64,92.51,93.91,18.78,0.71,11.5x
5,resnet50,PCA,32,84.06,87.34,17.47,0.63,12.9x
6,resnet50,PCA,16,67.24,74.24,14.85,0.65,12.6x


\begin{table}
\caption{PCA Reduction Performance for RESNET50}
\label{tab:dim_red_resnet50_PCA}
\begin{tabular}{llrrrrrl}
\toprule
Model & Method & Dim & mAP@10 & Prec@10 & Rec@10 & Time (ms) & Speedup \\
\midrule
resnet50 & RAW & 2048 & 100.00 & 100.00 & 100.00 & 8.14 & 1.0x \\
resnet50 & PCA & 512 & 98.40 & 98.63 & 19.73 & 5.18 & 1.6x \\
resnet50 & PCA & 256 & 97.54 & 97.96 & 19.59 & 3.22 & 2.5x \\
resnet50 & PCA & 128 & 95.95 & 96.66 & 19.33 & 0.77 & 10.6x \\
resnet50 & PCA & 64 & 92.51 & 93.91 & 18.78 & 0.71 & 11.5x \\
resnet50 & PCA & 32 & 84.06 & 87.34 & 17.47 & 0.63 & 12.9x \\
resnet50 & PCA & 16 & 67.24 & 74.24 & 14.85 & 0.65 & 12.6x \\
\bottomrule
\end{tabular}
\end{table}

Running GRP Pipeline for RESNET50 (vision)


,Model,Method,Dim,mAP@10,Prec@10,Rec@10,Time (ms),Speedup
0,resnet50,RAW,2048,100.00,100.00,100.00,7.84,1.0x
1,resnet50,GRP,512,98.94,99.15,19.83,2.91,2.7x
2,resnet50,GRP,256,95.82,96.70,19.34,4.25,1.8x
3,resnet50,GRP,128,85.13,88.43,17.68,0.56,14.1x
4,resnet50,GRP,64,68.50,75.39,15.08,0.42,18.5x
5,resnet50,GRP,32,41.98,52.60,10.52,0.35,22.3x
6,resnet50,GRP,16,21.30,32.07,6.41,0.32,24.8x


\begin{table}
\caption{GRP Reduction Performance for RESNET50}
\label{tab:dim_red_resnet50_GRP}
\begin{tabular}{llrrrrrl}
\toprule
Model & Method & Dim & mAP@10 & Prec@10 & Rec@10 & Time (ms) & Speedup \\
\midrule
resnet50 & RAW & 2048 & 100.00 & 100.00 & 100.00 & 7.84 & 1.0x \\
resnet50 & GRP & 512 & 98.94 & 99.15 & 19.83 & 2.91 & 2.7x \\
resnet50 & GRP & 256 & 95.82 & 96.70 & 19.34 & 4.25 & 1.8x \\
resnet50 & GRP & 128 & 85.13 & 88.43 & 17.68 & 0.56 & 14.1x \\
resnet50 & GRP & 64 & 68.50 & 75.39 & 15.08 & 0.42 & 18.5x \\
resnet50 & GRP & 32 & 41.98 & 52.60 & 10.52 & 0.35 & 22.3x \\
resnet50 & GRP & 16 & 21.30 & 32.07 & 6.41 & 0.32 & 24.8x \\
\bottomrule
\end{tabular}
\end{table}

Running PCA Pipeline for MOBILENET_V3 (vision)


...

\begin{table}
\caption{GRP Reduction Performance for GPT2}
\label{tab:dim_red_gpt2_GRP}
\begin{tabular}{llrrrrrl}
\toprule
Model & Method & Dim & mAP@10 & Prec@10 & Rec@10 & Time (ms) & Speedup \\
\midrule
gpt2 & RAW & 768 & 100.00 & 100.00 & 100.00 & 2.02 & 1.0x \\
gpt2 & GRP & 512 & 100.00 & 100.00 & 20.00 & 1.40 & 1.4x \\
gpt2 & GRP & 256 & 99.96 & 99.96 & 19.99 & 0.84 & 2.4x \\
gpt2 & GRP & 128 & 99.54 & 99.65 & 19.93 & 0.54 & 3.7x \\
gpt2 & GRP & 64 & 96.56 & 97.36 & 19.47 & 0.42 & 4.9x \\
gpt2 & GRP & 32 & 87.72 & 90.93 & 18.18 & 0.35 & 5.8x \\
gpt2 & GRP & 16 & 57.03 & 67.52 & 13.50 & 0.31 & 6.5x \\
\bottomrule
\end{tabular}
\end{table}

Running PCA Pipeline for CLIP_TEXT (text)


,Model,Method,Dim,mAP@10,Prec@10,Rec@10,Time (ms),Speedup
0,clip_text,RAW,512,100.00,100.00,100.00,1.40,1.0x
1,clip_text,PCA,256,99.58,99.65,19.93,1.08,1.3x
2,clip_text,PCA,128,99.15,99.31,19.86,0.56,2.5x
3,clip_text,PCA,64,97.11,97.77,19.55,0.61,2.3x
4,clip_text,PCA,32,88.83,91.59,18.32,0.34,4.1x
5,clip_text,PCA,16,71.80,78.88,15.77,0.52,2.7x


\begin{table}
\caption{PCA Reduction Performance for CLIP_TEXT}
\label{tab:dim_red_clip_text_PCA}
\begin{tabular}{llrrrrrl}
\toprule
Model & Method & Dim & mAP@10 & Prec@10 & Rec@10 & Time (ms) & Speedup \\
\midrule
clip_text & RAW & 512 & 100.00 & 100.00 & 100.00 & 1.40 & 1.0x \\
clip_text & PCA & 256 & 99.58 & 99.65 & 19.93 & 1.08 & 1.3x \\
clip_text & PCA & 128 & 99.15 & 99.31 & 19.86 & 0.56 & 2.5x \\
clip_text & PCA & 64 & 97.11 & 97.77 & 19.55 & 0.61 & 2.3x \\
clip_text & PCA & 32 & 88.83 & 91.59 & 18.32 & 0.34 & 4.1x \\
clip_text & PCA & 16 & 71.80 & 78.88 & 15.77 & 0.52 & 2.7x \\
\bottomrule
\end{tabular}
\end{table}

Running GRP Pipeline for CLIP_TEXT (text)


,Model,Method,Dim,mAP@10,Prec@10,Rec@10,Time (ms),Speedup
0,clip_text,RAW,512,100.00,100.00,100.00,1.33,1.0x
1,clip_text,GRP,256,97.81,98.23,19.64,0.85,1.6x
2,clip_text,GRP,128,91.81,93.58,18.72,0.54,2.4x
3,clip_text,GRP,64,74.79,80.35,16.07,0.42,3.1x
4,clip_text,GRP,32,50.19,59.99,12.00,0.35,3.9x
5,clip_text,GRP,16,28.28,39.82,7.96,0.31,4.3x


\begin{table}
\caption{GRP Reduction Performance for CLIP_TEXT}
\label{tab:dim_red_clip_text_GRP}
\begin{tabular}{llrrrrrl}
\toprule
Model & Method & Dim & mAP@10 & Prec@10 & Rec@10 & Time (ms) & Speedup \\
\midrule
clip_text & RAW & 512 & 100.00 & 100.00 & 100.00 & 1.33 & 1.0x \\
clip_text & GRP & 256 & 97.81 & 98.23 & 19.64 & 0.85 & 1.6x \\
clip_text & GRP & 128 & 91.81 & 93.58 & 18.72 & 0.54 & 2.4x \\
clip_text & GRP & 64 & 74.79 & 80.35 & 16.07 & 0.42 & 3.1x \\
clip_text & GRP & 32 & 50.19 & 59.99 & 12.00 & 0.35 & 3.9x \\
clip_text & GRP & 16 & 28.28 & 39.82 & 7.96 & 0.31 & 4.3x \\
\bottomrule
\end{tabular}
\end{table}


All experiments complete! Saved to MASTER_THESIS_RESULTS.csv
